# Arabic Story Retrieval-Augmented Generation (RAG) System

## Project Overview
This notebook builds an end-to-end **Retrieval-Augmented Generation (RAG)** system for a
collection of Arabic short stories. It covers the full pipeline: loading and cleaning raw
data, chunking text, building both keyword (BM25) and semantic (FAISS) search indexes,
combining them into a **Hybrid Search** retriever, evaluating retrieval quality against a
ground-truth query set, and finally generating grounded answers to user questions with an
open-source LLM.

## Dataset
The dataset (`arabic_stories.json`) is a collection of Arabic short stories. Each record
contains a `title` and the full `story` text. After cleaning, the stories are combined into
a single searchable corpus.

## Technology Stack
- **pandas / numpy** — data loading, cleaning, and manipulation
- **rank_bm25** — sparse, keyword-based (BM25) retrieval
- **sentence-transformers** — multilingual dense embeddings for semantic search
- **faiss** — fast approximate/exact nearest-neighbor search over embeddings
- **transformers / torch (Hugging Face)** — open-source causal LLM for answer generation

## Workflow

```
                ┌────────────────────┐
                │   Raw JSON Dataset  │
                └──────────┬──────────┘
                           │
                ┌──────────▼──────────┐
                │   Dataset Cleaning   │
                │  (dedupe, whitespace │
                │   normalization)     │
                └──────────┬──────────┘
                           │
                ┌──────────▼──────────┐
                │   Retrieval Text /   │
                │      Chunking        │
                └──────────┬──────────┘
                           │
              ┌────────────┴────────────┐
              │                         │
   ┌──────────▼──────────┐   ┌──────────▼──────────┐
   │      BM25 Index      │   │   Embeddings + FAISS │
   │   (keyword search)   │   │   (semantic search)  │
   └──────────┬──────────┘   └──────────┬──────────┘
              │                         │
              └────────────┬────────────┘
                           │
                ┌──────────▼──────────┐
                │    Hybrid Search      │
                │ (BM25 + FAISS scores) │
                └──────────┬──────────┘
                           │
                ┌──────────▼──────────┐
                │  Ground-Truth Eval    │
                │ (Precision/Recall/MRR)│
                └──────────┬──────────┘
                           │
                ┌──────────▼──────────┐
                │     Open-Source LLM   │
                └──────────┬──────────┘
                           │
                ┌──────────▼──────────┐
                │     RAG Pipeline       │
                │ (retrieve + generate)  │
                └──────────┬──────────┘
                           │
                ┌──────────▼──────────┐
                │   Answers to Queries   │
                └────────────────────┘
```


## 1. Import Libraries

Install the third-party packages this notebook depends on, then import everything needed
for data handling, retrieval (BM25 + FAISS), and embeddings.

In [ ]:
# Install required packages (Colab does not have these by default)
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q rank_bm25

In [ ]:
# Import core libraries
import json          # to load our JSON dataset
import re            # for text cleaning
import numpy as np   # for numerical operations
import pandas as pd  # for tabular data handling

# Import libraries for embeddings and retrieval
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi

# Print confirmation that all libraries loaded successfully
print("All libraries imported successfully!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

## 2. Load Dataset

Load the raw Arabic stories dataset from JSON into memory.

In [ ]:
# Define the path to our dataset file
# Update this path based on where you uploaded your file in Colab
dataset_path = "arabic_stories.json"

# Open and load the JSON file
with open(dataset_path, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

# Print confirmation that the dataset loaded successfully
print("Dataset loaded successfully!")
print("Type of raw_data:", type(raw_data))
print("Number of records loaded:", len(raw_data))

## 3. Dataset Exploration

Inspect the raw structure of the dataset, convert it into a flat `pandas` DataFrame, and
run some basic exploratory data analysis (shape, info, missing values, duplicates, and
story-length distribution) to understand what we're working with.

In [ ]:
# Print the type of the dataset
print("Dataset type:", type(raw_data))

# Print the number of stories in the dataset
print("Number of stories:", len(raw_data))

# Print the keys of the first story record
first_story = raw_data[0]
print("\nKeys in a story record:", list(first_story.keys()))

# Print a sample record to inspect its structure
print("\nSample record:")
print(first_story)

In [ ]:
# Convert the list of story dictionaries into a pandas DataFrame
df = pd.DataFrame(raw_data)

# Keep only the two columns that actually exist in our dataset
df = df[["title", "story"]]

# Print confirmation
print("DataFrame created successfully!")
print("Shape of DataFrame:", df.shape)
print("\nColumn names:", list(df.columns))

In [ ]:
# Preview the first few rows to verify the data looks correct
df.head()

In [ ]:
# 1. Shape of the dataset
print("Shape of dataset (rows, columns):", df.shape)

In [ ]:
# 2. General info about the DataFrame
df.info()

In [ ]:
# 3. Descriptive statistics (mostly useful for text length, since our columns are text)
df.describe(include="all")

In [ ]:
# 4. Check for missing values in each column
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# 5. Check for fully duplicate rows (same title AND same story)
duplicate_rows = df.duplicated().sum()
print("Number of fully duplicate rows:", duplicate_rows)

In [ ]:
# 6. Check for duplicate stories (same story text, possibly different titles)
duplicate_stories = df.duplicated(subset=["story"]).sum()
print("Number of duplicate stories (by story text):", duplicate_stories)

In [ ]:
# 7. Word Count Statistics
# We compute word counts temporarily for analysis only.
# This is NOT stored as a new dataset column — it's just a helper Series for EDA.
temp_word_counts = df["story"].apply(lambda x: len(str(x).split()))

print("Word Count Statistics:")
print(temp_word_counts.describe())

In [ ]:
# 8. Story Length Distribution
# We use pandas' built-in histogram-style value counts on binned word counts

bins = [0, 100, 200, 300, 500, 1000, 2000, 5000, 100000]
labels = ["0-100", "100-200", "200-300", "300-500", "500-1000", "1000-2000", "2000-5000", "5000+"]

length_distribution = pd.cut(temp_word_counts, bins=bins, labels=labels).value_counts().sort_index()

print("Story Length Distribution (word count ranges):")
print(length_distribution)

## 4. Data Cleaning

Remove duplicate and empty stories, normalize whitespace/newlines, and strip any leftover
Project Gutenberg boilerplate that might be present in the raw text.

In [ ]:
# Step 1: Remove duplicate stories (keep the first occurrence)
print("Rows before removing duplicate stories:", df.shape[0])

df = df.drop_duplicates(subset=["story"], keep="first")

print("Rows after removing duplicate stories:", df.shape[0])

In [ ]:
# Step 2 & 3: Remove missing or empty stories

# Remove rows where story is NaN
df = df.dropna(subset=["story"])

# Remove rows where story is empty or only whitespace
df = df[df["story"].str.strip() != ""]

print("Rows after removing missing/empty stories:", df.shape[0])

In [ ]:
# Step 4, 5, 6: Text cleaning function
# This function normalizes whitespace, removes extra newlines, and strips spaces

def clean_text(text):
    """
    Cleans a story text by:
    - Replacing multiple newlines with a single space
    - Replacing multiple spaces/tabs with a single space
    - Stripping leading and trailing whitespace

    Parameters:
        text (str): raw story text

    Returns:
        str: cleaned story text
    """
    # Replace newlines (single or multiple) with a space
    text = re.sub(r"\n+", " ", text)

    # Replace multiple spaces or tabs with a single space
    text = re.sub(r"\s+", " ", text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text


# Apply the cleaning function to the story column
df["story"] = df["story"].apply(clean_text)

print("Whitespace and newline cleaning applied successfully!")

In [ ]:
# Step 7: Remove Gutenberg-style header/footer if found
# This is a general safety check in case any boilerplate text exists.
# Common Gutenberg markers include phrases like "START OF THIS PROJECT GUTENBERG" or "END OF THIS PROJECT GUTENBERG"

def remove_gutenberg_boilerplate(text):
    """
    Removes Project Gutenberg header/footer boilerplate if present in the text.

    Parameters:
        text (str): story text

    Returns:
        str: story text with Gutenberg boilerplate removed (if any was found)
    """
    # Pattern for Gutenberg start marker
    start_pattern = r"\*\*\*\s*START OF (THIS|THE) PROJECT GUTENBERG.*?\*\*\*"
    # Pattern for Gutenberg end marker
    end_pattern = r"\*\*\*\s*END OF (THIS|THE) PROJECT GUTENBERG.*?\*\*\*"

    # Remove everything before and including the start marker (if found)
    text = re.split(start_pattern, text, flags=re.IGNORECASE)[-1]

    # Remove everything after and including the end marker (if found)
    text = re.split(end_pattern, text, flags=re.IGNORECASE)[0]

    return text.strip()


# Apply the Gutenberg cleaning function
df["story"] = df["story"].apply(remove_gutenberg_boilerplate)

print("Gutenberg boilerplate check applied successfully!")

In [ ]:
# Step 8: Verify cleaning results

print("Final shape after cleaning:", df.shape)

print("\nMissing values check:")
print(df.isnull().sum())

print("\nAny remaining empty stories:", (df["story"].str.strip() == "").sum())

print("\nSample cleaned story (first 300 characters):")
print(df["story"].iloc[0][:300])

## 5. Create Retrieval Text

Combine each story's `title` and `story` body into a single `retrieval_text` field. This
is the text that will actually be chunked and indexed for retrieval.

In [ ]:
# Function to build a readable retrieval_text from title and story
def build_retrieval_text(row):
    """
    Combines the title and story into a single readable text block.

    Parameters:
        row (pd.Series): a row from the DataFrame containing 'title' and 'story'

    Returns:
        str: formatted retrieval text
    """
    retrieval_text = f"Title: {row['title']}\n\nStory:\n{row['story']}"
    return retrieval_text


# Apply the function to every row to create the retrieval_text column
df["retrieval_text"] = df.apply(build_retrieval_text, axis=1)

print("retrieval_text column created successfully!")

In [ ]:
# Verify the new column
print("Updated DataFrame columns:", list(df.columns))
print("\nShape of DataFrame:", df.shape)

print("\nSample retrieval_text (first 400 characters):")
print(df["retrieval_text"].iloc[0][:400])

## 6. Chunking

Split each story's `retrieval_text` into overlapping, fixed-size chunks (paragraph →
sentence → word fallback) so that retrieval can operate at the chunk level instead of the
whole-document level.

In [ ]:
def split_into_paragraphs(text):
    """
    Splits text into paragraphs using double newlines or single newlines as separators.

    Parameters:
        text (str): input text

    Returns:
        list: list of paragraph strings
    """
    paragraphs = re.split(r"\n+", text)
    paragraphs = [p.strip() for p in paragraphs if p.strip() != ""]
    return paragraphs


def split_into_sentences(text):
    """
    Splits text into sentences using basic punctuation rules.
    Works reasonably for both English and Arabic text.

    Parameters:
        text (str): input text

    Returns:
        list: list of sentence strings
    """
    # Split on '.', '!', '?', or Arabic sentence-ending punctuation '؟' and '،' is NOT used (comma)
    sentences = re.split(r"(?<=[.!?؟])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip() != ""]
    return sentences


def split_into_words(text):
    """
    Splits text into a list of words using whitespace.

    Parameters:
        text (str): input text

    Returns:
        list: list of word strings
    """
    return text.split()

In [ ]:
def recursive_chunk_text(text, chunk_size=400, overlap=80):
    """
    Recursively splits text into overlapping chunks of a target word count.

    Strategy:
    1. Split text into paragraphs.
    2. Build chunks by adding paragraphs until the word limit is reached.
    3. If a single paragraph is bigger than chunk_size, split it into sentences.
    4. If a single sentence is still bigger than chunk_size, split it into words.
    5. Apply word-level overlap between consecutive chunks.

    Parameters:
        text (str): the full text to chunk (usually retrieval_text)
        chunk_size (int): target number of words per chunk
        overlap (int): number of overlapping words between consecutive chunks

    Returns:
        list: list of chunk strings
    """

    # Step 1: Break text into paragraphs
    paragraphs = split_into_paragraphs(text)

    # Step 2: Flatten paragraphs into a single list of "pieces" (words),
    # while respecting paragraph/sentence boundaries where possible
    all_words = []

    for paragraph in paragraphs:
        paragraph_words = split_into_words(paragraph)

        # If a paragraph is too long on its own, break it into sentences first
        if len(paragraph_words) > chunk_size:
            sentences = split_into_sentences(paragraph)
            for sentence in sentences:
                sentence_words = split_into_words(sentence)
                all_words.extend(sentence_words)
        else:
            all_words.extend(paragraph_words)

    # Step 3: Build overlapping chunks from the flat word list
    chunks = []
    start = 0
    total_words = len(all_words)

    while start < total_words:
        end = start + chunk_size
        chunk_words = all_words[start:end]
        chunk_text = " ".join(chunk_words)
        chunks.append(chunk_text)

        # Move start forward by (chunk_size - overlap) to create overlap
        start += (chunk_size - overlap)

    return chunks

In [ ]:
# Quick test of the chunking function on a single story before applying it to the whole dataset

sample_text = df["retrieval_text"].iloc[0]
sample_chunks = recursive_chunk_text(sample_text, chunk_size=400, overlap=80)

print("Number of chunks created for sample story:", len(sample_chunks))
print("\nFirst chunk preview:")
print(sample_chunks[0][:300])

print("\nWord count of first chunk:", len(sample_chunks[0].split()))

In [ ]:
# List to collect all chunk records before building the final DataFrame
chunk_records = []

# Global counter to assign a unique chunk_id across the whole dataset
global_chunk_id = 0

# Loop through every story in the cleaned DataFrame
for document_id, row in df.reset_index(drop=True).iterrows():

    # Generate chunks for this story's retrieval_text
    story_chunks = recursive_chunk_text(row["retrieval_text"], chunk_size=400, overlap=80)

    # Create a record for each chunk
    for chunk_text in story_chunks:
        chunk_records.append({
            "document_id": document_id,
            "chunk_id": global_chunk_id,
            "chunk_text": chunk_text,
            "title": row["title"]
        })
        global_chunk_id += 1

print("Total chunks created:", len(chunk_records))

In [ ]:
# Build the chunks DataFrame from the list of chunk records
chunks_df = pd.DataFrame(chunk_records)

# Print shape of the chunks DataFrame
print("Shape of chunks_df:", chunks_df.shape)
# Preview the first few rows of the chunks DataFrame
chunks_df.head()

In [ ]:
# Calculate average chunk length (in words)
chunks_df["chunk_word_count"] = chunks_df["chunk_text"].apply(lambda x: len(x.split()))

average_chunk_length = chunks_df["chunk_word_count"].mean()

print("Average chunk length (in words):", round(average_chunk_length, 2))
print("\nChunk word count statistics:")
print(chunks_df["chunk_word_count"].describe())

## 7. Build BM25 Index

Tokenize every chunk and build a sparse **BM25** index for keyword-based retrieval.

In [ ]:
def simple_tokenize(text):
    """
    A simple, educational tokenizer for Arabic and English text.

    Steps:
    - Convert text to lowercase (affects English letters; Arabic has no case)
    - Remove extra spaces
    - Extract Arabic words, English words, and numbers using regex

    Notes:
    - Does NOT remove Arabic stopwords
    - Does NOT perform stemming
    - Does NOT perform lemmatization
    - Punctuation is removed only as a side effect of the regex extraction

    Parameters:
        text (str): input text

    Returns:
        list: list of token strings
    """
    # Convert to lowercase (no effect on Arabic characters, but normalizes English)
    text = text.lower()

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Extract Arabic words, English words, and numbers
    # \u0600-\u06FF covers the Arabic Unicode block
    tokens = re.findall(r"[\u0600-\u06FF]+|[a-zA-Z]+|\d+", text)

    return tokens

In [ ]:
# Quick test of the tokenizer on a sample chunk
sample_chunk = chunks_df["chunk_text"].iloc[0]
sample_tokens = simple_tokenize(sample_chunk)

print("Sample tokenizer test:")
print("Original text (first 150 chars):", sample_chunk[:150])
print("\nTokenized output (first 20 tokens):", sample_tokens[:20])

In [ ]:
# Tokenize every chunk in chunks_df using our simple tokenizer
tokenized_documents = [simple_tokenize(text) for text in chunks_df["chunk_text"]]

print("Tokenization completed successfully!")
print("Number of tokenized documents:", len(tokenized_documents))

In [ ]:
# Build the BM25 index using rank_bm25
bm25 = BM25Okapi(tokenized_documents)

print("BM25 index built successfully!")

# Print number of indexed chunks
print("Number of indexed chunks:", len(tokenized_documents))

# Print average number of tokens per chunk
avg_tokens = sum(len(doc) for doc in tokenized_documents) / len(tokenized_documents)
print("Average number of tokens per chunk:", round(avg_tokens, 2))

# Print first tokenized example
print("\nFirst tokenized example (first 20 tokens):")
print(tokenized_documents[0][:20])

In [ ]:
def retrieve_top_k_bm25(query, documents, bm25, k=5):
    """
    Retrieves the top-k most relevant chunks for a query using BM25.

    Parameters:
        query (str): the search query
        documents (list): list of tokenized documents (same order used to build bm25)
        bm25 (BM25Okapi): the fitted BM25 index
        k (int): number of top results to return

    Returns:
        top_indices (list): indices of the top-k chunks (in chunks_df)
        top_scores (list): corresponding BM25 scores
    """
    # Tokenize the query using the same simple tokenizer
    tokenized_query = simple_tokenize(query)

    # Calculate BM25 scores for the query against all documents
    scores = bm25.get_scores(tokenized_query)

    # Get indices of the top-k scores, sorted from highest to lowest
    top_indices = np.argsort(scores)[::-1][:k]

    # Get the corresponding scores
    top_scores = [scores[i] for i in top_indices]

    return list(top_indices), top_scores

In [ ]:
# Test BM25 retrieval with 5 Arabic queries

test_queries = [
    "ساعة قديمة توقفت في القرية",
    "فتاة صغيرة تكتشف سراً",
    "قصة عن الزمن والحياة",
    "مهرجان الربيع في القرية",
    "برج الكنيسة والجرس"
]

for query in test_queries:
    print("=" * 80)
    print("Query:", query)

    top_indices, top_scores = retrieve_top_k_bm25(query, tokenized_documents, bm25, k=1)

    for idx, score in zip(top_indices, top_scores):
        retrieved_title = chunks_df["title"].iloc[idx]
        chunk_preview = chunks_df["chunk_text"].iloc[idx][:200]

        print("\nRetrieved chunk title:", retrieved_title)
        print("BM25 score:", round(score, 4))
        print("Chunk preview:", chunk_preview)

print("=" * 80)
print("\nBM25 testing completed!")

## 8. Generate Embeddings

Load a pretrained multilingual sentence-embedding model and generate a dense vector
embedding for every chunk, to support semantic (meaning-based) retrieval.

In [ ]:
# Load the pretrained multilingual sentence embedding model
# Switched from all-MiniLM-L6-v2 to a multilingual model since our dataset is in Arabic
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(model_name)

print("Embedding model loaded successfully:", model_name)

In [ ]:
# Generate embeddings for all chunk texts
# show_progress_bar helps track progress on larger datasets

chunk_texts = chunks_df["chunk_text"].tolist()

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    normalize_embeddings=True  # normalize so we can use cosine similarity with IndexFlatIP
)

print("Embeddings generated successfully!")
print("Shape of embeddings array:", embeddings.shape)

## 9. Build FAISS Index

Index the chunk embeddings in a **FAISS** index (Inner Product / cosine similarity) and
implement the semantic top-k retrieval function used by Hybrid Search.

In [ ]:
# Get the embedding dimension (number of features per vector)
embedding_dimension = embeddings.shape[1]

# Create a FAISS index using Inner Product (IP) similarity
faiss_index = faiss.IndexFlatIP(embedding_dimension)

# Add our embeddings to the index
faiss_index.add(embeddings)

print("FAISS index built successfully!")

In [ ]:
# Print index details for verification
print("Embedding Dimension:", embedding_dimension)
print("Number of vectors in FAISS index:", faiss_index.ntotal)

In [ ]:
def retrieve_top_k_faiss(query, faiss_index, embedding_model, k=5):
    """
    Retrieves the top-k most relevant chunks for a query using FAISS (semantic search).

    Parameters:
        query (str): the search query
        faiss_index (faiss.Index): the built FAISS index
        embedding_model (SentenceTransformer): the model used to embed chunks
        k (int): number of top results to return

    Returns:
        top_indices (list): indices of the top-k chunks (in chunks_df)
        top_scores (list): corresponding similarity scores (Inner Product / Cosine)
    """
    # Embed the query using the same model and normalization as our chunks
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)

    # Search the FAISS index for the top-k most similar vectors
    scores, indices = faiss_index.search(query_embedding, k)

    # Extract the first (and only) row of results, since we passed a single query
    top_indices = indices[0].tolist()
    top_scores = scores[0].tolist()

    return top_indices, top_scores

In [ ]:
# Quick test of retrieve_top_k_faiss() using one of our earlier test queries

test_query = "ساعة قديمة توقفت في القرية"

faiss_indices, faiss_scores = retrieve_top_k_faiss(test_query, faiss_index, embedding_model, k=3)

print("Query:", test_query)
print("\nTop FAISS results:")

for idx, score in zip(faiss_indices, faiss_scores):
    retrieved_title = chunks_df["title"].iloc[idx]
    chunk_preview = chunks_df["chunk_text"].iloc[idx][:200]

    print("\nChunk index:", idx)
    print("Title:", retrieved_title)
    print("FAISS score:", round(score, 4))
    print("Chunk preview:", chunk_preview)

## 10. Hybrid Search

Combine the BM25 (keyword) and FAISS (semantic) retrieval scores into a single **Hybrid
Search** ranking, then test it against a broader set of example queries.

In [ ]:
def normalize_scores(scores):
    """
    Normalizes a list of scores to a 0-1 range using min-max normalization.

    Parameters:
        scores (list): list of raw scores

    Returns:
        list: normalized scores between 0 and 1
    """
    scores = np.array(scores, dtype=float)

    # Avoid division by zero if all scores are the same
    if scores.max() == scores.min():
        return [1.0 for _ in scores]

    normalized = (scores - scores.min()) / (scores.max() - scores.min())
    return normalized.tolist()

In [ ]:
def hybrid_search(query, faiss_index, embedding_model, bm25, tokenized_documents, chunks_df,
                   k=5, alpha=0.7, retrieval_pool=20):
    """
    Performs Hybrid Search by combining FAISS (semantic) and BM25 (keyword) scores.

    Hybrid Score = alpha * normalized_FAISS_score + (1 - alpha) * normalized_BM25_score

    Parameters:
        query (str): the search query
        faiss_index (faiss.Index): the built FAISS index
        embedding_model (SentenceTransformer): model used to embed the query
        bm25 (BM25Okapi): the fitted BM25 index
        tokenized_documents (list): tokenized chunks used to build BM25
        chunks_df (pd.DataFrame): DataFrame containing chunk_text, title, etc.
        k (int): number of final top results to return
        alpha (float): weight given to FAISS score (BM25 gets 1 - alpha)
        retrieval_pool (int): how many candidates to pull from each method before merging

    Returns:
        pd.DataFrame: top-k merged results sorted by hybrid_score, with columns:
                       chunk_index, title, chunk_text, faiss_score, bm25_score, hybrid_score
    """

    # Step 1: Retrieve a larger candidate pool from FAISS
    faiss_indices, faiss_scores = retrieve_top_k_faiss(query, faiss_index, embedding_model, k=retrieval_pool)

    # Step 2: Retrieve a larger candidate pool from BM25
    bm25_indices, bm25_scores = retrieve_top_k_bm25(query, tokenized_documents, bm25, k=retrieval_pool)

    # Step 3: Normalize both score sets independently
    norm_faiss_scores = normalize_scores(faiss_scores)
    norm_bm25_scores = normalize_scores(bm25_scores)

    # Step 4: Merge scores by chunk index using a dictionary
    # Each entry stores {chunk_index: {"faiss": score, "bm25": score}}
    merged_scores = {}

    for idx, score in zip(faiss_indices, norm_faiss_scores):
        merged_scores[idx] = {"faiss": score, "bm25": 0.0}

    for idx, score in zip(bm25_indices, norm_bm25_scores):
        if idx in merged_scores:
            merged_scores[idx]["bm25"] = score
        else:
            merged_scores[idx] = {"faiss": 0.0, "bm25": score}

    # Step 5: Compute the hybrid score for every merged chunk
    results = []
    for idx, score_dict in merged_scores.items():
        hybrid_score = (alpha * score_dict["faiss"]) + ((1 - alpha) * score_dict["bm25"])
        results.append({
            "chunk_index": idx,
            "title": chunks_df["title"].iloc[idx],
            "chunk_text": chunks_df["chunk_text"].iloc[idx],
            "faiss_score": score_dict["faiss"],
            "bm25_score": score_dict["bm25"],
            "hybrid_score": hybrid_score
        })

    # Step 6: Sort by hybrid_score, descending
    results_df = pd.DataFrame(results).sort_values(by="hybrid_score", ascending=False)

    # Step 7: Return only the top-k results
    return results_df.head(k).reset_index(drop=True)

In [ ]:
# Quick test of hybrid_search() using one of our earlier test queries

test_query = "ساعة قديمة توقفت في القرية"

hybrid_results = hybrid_search(
    query=test_query,
    faiss_index=faiss_index,
    embedding_model=embedding_model,
    bm25=bm25,
    tokenized_documents=tokenized_documents,
    chunks_df=chunks_df,
    k=5,
    alpha=0.7
)

print("Query:", test_query)
print("\nHybrid Search Results:")
hybrid_results

In [ ]:
# Prepare 10 example Arabic test queries covering different topics/themes
final_test_queries = [
    "ساعة قديمة توقفت في القرية",
    "فتاة صغيرة تكتشف سراً",
    "قصة عن الزمن والحياة",
    "مهرجان الربيع في القرية",
    "برج الكنيسة والجرس",
    "قصة عن الصداقة والوفاء",
    "حكاية عن الغابة والحيوانات",
    "طفل يتعلم درساً مهماً في الحياة",
    "قصة عن الشجاعة والمغامرة",
    "حكمة من قصص الأجداد"
]

print("Prepared", len(final_test_queries), "test queries for final retrieval testing.")

In [ ]:
# Run Hybrid Search for each test query and print the results

for query in final_test_queries:
    print("=" * 100)
    print("Query:", query)

    results = hybrid_search(
        query=query,
        faiss_index=faiss_index,
        embedding_model=embedding_model,
        bm25=bm25,
        tokenized_documents=tokenized_documents,
        chunks_df=chunks_df,
        k=3,
        alpha=0.7
    )

    # Print each retrieved result for this query
    for row_num, row in results.iterrows():
        print(f"\nResult {row_num + 1}:")
        print("Retrieved Story Title:", row["title"])
        print("Hybrid Score:", round(row["hybrid_score"], 4))
        print("Chunk Preview:", row["chunk_text"][:200])

print("=" * 100)
print("\nFinal retrieval testing completed successfully!")

## 11. Ground Truth

Build a ground-truth evaluation set that maps realistic Arabic queries (by title, topic,
event, character, and moral/lesson) to the exact story title each query should retrieve.

In [ ]:
# First, let's print all available story titles so we can build realistic ground truth queries
print("Total number of stories:", df["title"].nunique())
print("\nSample of available titles:")
for title in df["title"].unique()[:20]:
    print("-", title)

In [ ]:
# Build the ground truth evaluation set.
# Each row links a realistic Arabic query to the exact title of the story it should retrieve.
#
# Query types included (as required):
# - searching by story title
# - searching by story topic
# - searching by story events
# - searching by important characters
# - searching by lesson or moral
#
# NOTE: The rows below using "الساعة التي توقفت في لحظة فارقة" are fully worked examples.
# Extend this list to 20-30 rows using titles printed above from YOUR dataset,
# following the same query-type pattern.

ground_truth_data = [
    # --- Searching by story title (exact or close to title) ---
    {"query": "الساعة التي توقفت في لحظة فارقة", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},
    {"query": "قصة الساعة التي توقفت", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},

    # --- Searching by story topic ---
    {"query": "قصة عن ساعة قديمة في قرية", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},
    {"query": "حكاية عن الزمن والوقت", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},

    # --- Searching by story events ---
    {"query": "ساعة توقفت في برج الكنيسة أثناء مهرجان الربيع", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},
    {"query": "قرية تحتفل بمهرجان الربيع وساعتها معطلة", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},

    # --- Searching by important characters ---
    {"query": "فتاة تدعى ليلى تتسلق برج الساعة", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},
    {"query": "من هي ليلى التي أصلحت الساعة؟", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},

    # --- Searching by lesson or moral ---
    {"query": "قصة تعلم أهمية الاستمتاع باللحظات الجميلة", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},
    {"query": "حكمة عن أن الحياة لا تتوقف مهما حدث", "ground_truth_title": "الساعة التي توقفت في لحظة فارقة"},

    # Additional rows can be added here following the same query-type pattern above,
    # using more titles from the dataset.
]

# Build the ground_truth_df DataFrame
ground_truth_df = pd.DataFrame(ground_truth_data)

print("ground_truth_df created successfully!")
print("Shape:", ground_truth_df.shape)

In [ ]:
# Preview the ground truth DataFrame
ground_truth_df.head(10)

## 12. Evaluation

Implement standard retrieval metrics (Precision@k, Recall@k, Hit Rate@k, and Mean
Reciprocal Rank) and run Hybrid Search over the ground-truth set to measure retrieval
quality.

In [ ]:
def precision_at_k(retrieved_titles, ground_truth_title, k=5):
    """
    Calculates Precision@k: fraction of the top-k retrieved titles that match the ground truth title.

    Parameters:
        retrieved_titles (list): list of titles retrieved by Hybrid Search (already limited to top-k)
        ground_truth_title (str): the correct title for this query
        k (int): number of retrieved items considered

    Returns:
        float: precision score between 0 and 1
    """
    # Count how many of the top-k retrieved titles match the ground truth
    relevant_count = sum(1 for title in retrieved_titles[:k] if title == ground_truth_title)

    return relevant_count / k

In [ ]:
def recall_at_k(retrieved_titles, ground_truth_title, k=5):
    """
    Calculates Recall@k: whether the correct title was retrieved within the top-k.
    Since each query has exactly ONE correct story, recall is either 0 or 1.

    Parameters:
        retrieved_titles (list): list of titles retrieved by Hybrid Search (already limited to top-k)
        ground_truth_title (str): the correct title for this query
        k (int): number of retrieved items considered

    Returns:
        float: recall score, 0.0 or 1.0
    """
    # Check if the ground truth title appears anywhere in the top-k retrieved titles
    if ground_truth_title in retrieved_titles[:k]:
        return 1.0
    else:
        return 0.0

In [ ]:
def hit_rate_at_k(retrieved_titles, ground_truth_title, k=5):
    """
    Calculates Hit Rate@k: 1 if the correct title appears anywhere in the top-k, else 0.
    (Conceptually similar to recall here, but kept as a separate function since it's a
    standard, distinct metric name in retrieval evaluation.)

    Parameters:
        retrieved_titles (list): list of titles retrieved by Hybrid Search (already limited to top-k)
        ground_truth_title (str): the correct title for this query
        k (int): number of retrieved items considered

    Returns:
        float: 1.0 if hit, 0.0 otherwise
    """
    if ground_truth_title in retrieved_titles[:k]:
        return 1.0
    else:
        return 0.0

In [ ]:
def mean_reciprocal_rank(retrieved_titles, ground_truth_title, k=5):
    """
    Calculates the Reciprocal Rank for a single query: 1 / (rank of first correct match).
    If the correct title is not found within the top-k, returns 0.

    Parameters:
        retrieved_titles (list): list of titles retrieved by Hybrid Search (already limited to top-k)
        ground_truth_title (str): the correct title for this query
        k (int): number of retrieved items considered

    Returns:
        float: reciprocal rank score between 0 and 1
    """
    # Loop through the retrieved titles and find the rank (1-indexed) of the first correct match
    for rank, title in enumerate(retrieved_titles[:k], start=1):
        if title == ground_truth_title:
            return 1.0 / rank

    # If no match was found within top-k, reciprocal rank is 0
    return 0.0

In [ ]:
# Run evaluation: for every query in ground_truth_df, perform Hybrid Search
# and compute Precision@5, Recall@5, Hit Rate@5, and Reciprocal Rank

evaluation_records = []

K = 5  # evaluation cutoff

for _, row in ground_truth_df.iterrows():
    query = row["query"]
    ground_truth_title = row["ground_truth_title"]

    # Run Hybrid Search for this query
    search_results = hybrid_search(
        query=query,
        faiss_index=faiss_index,
        embedding_model=embedding_model,
        bm25=bm25,
        tokenized_documents=tokenized_documents,
        chunks_df=chunks_df,
        k=K,
        alpha=0.7
    )

    # Get the list of retrieved titles (in ranked order)
    retrieved_titles = search_results["title"].tolist()

    # Compute all four metrics for this query
    precision = precision_at_k(retrieved_titles, ground_truth_title, k=K)
    recall = recall_at_k(retrieved_titles, ground_truth_title, k=K)
    hit_rate = hit_rate_at_k(retrieved_titles, ground_truth_title, k=K)
    reciprocal_rank = mean_reciprocal_rank(retrieved_titles, ground_truth_title, k=K)

    # Store this query's evaluation results
    evaluation_records.append({
        "query": query,
        "ground_truth_title": ground_truth_title,
        "precision_at_5": precision,
        "recall_at_5": recall,
        "hit_rate_at_5": hit_rate,
        "reciprocal_rank": reciprocal_rank
    })

print("Evaluation completed for", len(evaluation_records), "queries.")

In [ ]:
# Build the evaluation results DataFrame
evaluation_df = pd.DataFrame(evaluation_records)

# Compute average metrics across all queries
average_precision = evaluation_df["precision_at_5"].mean()
average_recall = evaluation_df["recall_at_5"].mean()
average_hit_rate = evaluation_df["hit_rate_at_5"].mean()
average_mrr = evaluation_df["reciprocal_rank"].mean()

print("Average Precision@5:", round(average_precision, 4))
print("Average Recall@5:", round(average_recall, 4))
print("Average Hit Rate@5:", round(average_hit_rate, 4))
print("Average MRR:", round(average_mrr, 4))

# Display the complete evaluation table
evaluation_df

## 13. Load LLM

Load an open-source causal language model (with an automatic fallback if the primary
model fails to load) that will generate answers grounded in the retrieved context.

In [ ]:
# Install/upgrade transformers if needed (usually pre-installed on Colab)
!pip install -q -U transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Check if a GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
def load_llm(model_name):
    """
    Loads a Hugging Face tokenizer and causal language model.

    Parameters:
        model_name (str): Hugging Face model identifier

    Returns:
        tokenizer, model: the loaded tokenizer and model objects
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None
    )

    if device == "cpu":
        model = model.to(device)

    return tokenizer, model

In [ ]:
# Try loading the primary model first. If it fails for any reason,
# automatically fall back to the same lightweight model as a safety net.

primary_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
fallback_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

try:
    print("Attempting to load primary model:", primary_model_name)
    llm_tokenizer, llm_model = load_llm(primary_model_name)
    loaded_model_name = primary_model_name
    print("Primary model loaded successfully!")

except Exception as error:
    print("Primary model failed to load. Reason:", str(error))
    print("Falling back to smaller model:", fallback_model_name)

    llm_tokenizer, llm_model = load_llm(fallback_model_name)
    loaded_model_name = fallback_model_name
    print("Fallback model loaded successfully!")

print("\nFinal model in use:", loaded_model_name)

## 14. RAG Pipeline

Assemble the full Retrieval-Augmented Generation pipeline: Hybrid Search retrieves
relevant chunks, a strict prompt template constrains the LLM to answer only from that
context, and `generate_rag_answer()` ties retrieval and generation together. A
`safe_generate_rag_answer()` wrapper adds basic error handling (empty/short queries,
retrieval failures, and generation failures).

In [ ]:
# No code needed for this section — it's a planning/overview step.
# Simply printing the pipeline stages for clarity before we start implementing them.

pipeline_stages = [
    "User Question",
    "Hybrid Search",
    "Top-K Chunks",
    "Prompt Construction",
    "LLM",
    "Answer"
]

print("RAG Pipeline Stages:")
for stage_number, stage in enumerate(pipeline_stages, start=1):
    print(f"{stage_number}. {stage}")

In [ ]:
def build_rag_prompt(context, question):
    """
    Builds a strict RAG prompt that instructs the model to answer only from the given context.

    Parameters:
        context (str): the retrieved chunk text(s) to use as the only source of truth
        question (str): the user's question

    Returns:
        str: the fully formatted prompt
    """
    prompt = f"""أنت مساعد يجيب فقط بناءً على السياق المُعطى من القصص التالية.

القواعد:
- أجب فقط باستخدام المعلومات الموجودة في السياق أدناه.
- لا تخترع أي معلومات غير موجودة في السياق.
- لا تستخدم أي معرفة خارجية.
- إذا لم تجد إجابة كافية في السياق، أجب بالضبط بهذه الجملة:
"لم أجد معلومات كافية للإجابة اعتمادًا على القصص الموجودة."

السياق:
{context}

السؤال:
{question}

الإجابة:"""

    return prompt

In [ ]:
# Quick test of the prompt template with dummy context and question
sample_context = "في قرية صغيرة تقع بين الجبال، كانت هناك ساعة قديمة معلقة في برج الكنيسة."
sample_question = "أين كانت الساعة معلقة؟"

sample_prompt = build_rag_prompt(sample_context, sample_question)

print("Sample prompt preview:")
print(sample_prompt)

In [ ]:
def generate_rag_answer(user_question, k=5, max_new_tokens=200):
    """
    Full RAG pipeline: retrieves relevant chunks and generates an answer using the LLM.

    Parameters:
        user_question (str): the question asked by the user
        k (int): number of top chunks to retrieve via Hybrid Search
        max_new_tokens (int): maximum number of tokens the LLM should generate

    Returns:
        dict: {
            "answer": generated answer text,
            "retrieved_titles": list of story titles used as context,
            "retrieved_chunks": list of chunk texts used as context,
            "hybrid_scores": list of hybrid scores for the retrieved chunks
        }
    """
    # Step 1: Retrieve top-k chunks using Hybrid Search
    search_results = hybrid_search(
        query=user_question,
        faiss_index=faiss_index,
        embedding_model=embedding_model,
        bm25=bm25,
        tokenized_documents=tokenized_documents,
        chunks_df=chunks_df,
        k=k,
        alpha=0.7
    )

    retrieved_titles = search_results["title"].tolist()
    retrieved_chunks = search_results["chunk_text"].tolist()
    hybrid_scores = search_results["hybrid_score"].tolist()

    # Step 2: Build context by joining retrieved chunks together
    context = "\n\n---\n\n".join(retrieved_chunks)

    # Step 3: Build the final prompt
    prompt = build_rag_prompt(context, user_question)

    # Step 4: Tokenize the prompt and generate an answer using the LLM
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)

    output_tokens = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=llm_tokenizer.eos_token_id
    )

    # Step 5: Decode only the newly generated tokens (exclude the input prompt itself)
    generated_tokens = output_tokens[0][inputs["input_ids"].shape[1]:]
    answer = llm_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Step 6: Return everything needed for inspection/testing
    return {
        "answer": answer,
        "retrieved_titles": retrieved_titles,
        "retrieved_chunks": retrieved_chunks,
        "hybrid_scores": hybrid_scores
    }

In [ ]:
# Quick test of the full RAG pipeline
test_result = generate_rag_answer("أين كانت الساعة معلقة؟", k=3)

print("Generated Answer:", test_result["answer"])
print("\nRetrieved Titles:", test_result["retrieved_titles"])
print("Hybrid Scores:", [round(s, 4) for s in test_result["hybrid_scores"]])

In [ ]:
def safe_generate_rag_answer(user_question, k=5, max_new_tokens=200, min_query_length=3):
    """
    A safer wrapper around generate_rag_answer() that handles common error cases gracefully.

    Handles:
    - Empty query
    - Very short query
    - No retrieved chunks
    - Model generation failure

    Parameters:
        user_question (str): the question asked by the user
        k (int): number of top chunks to retrieve via Hybrid Search
        max_new_tokens (int): maximum number of tokens the LLM should generate
        min_query_length (int): minimum number of characters required for a valid query

    Returns:
        dict: same structure as generate_rag_answer(), or a friendly error message in "answer"
    """

    # Case 1: Empty query
    if user_question is None or user_question.strip() == "":
        print("⚠️ Empty query received.")
        return {
            "answer": "يرجى إدخال سؤال صحيح.",
            "retrieved_titles": [],
            "retrieved_chunks": [],
            "hybrid_scores": []
        }

    # Case 2: Very short query
    if len(user_question.strip()) < min_query_length:
        print("⚠️ Query is too short to process.")
        return {
            "answer": "سؤالك قصير جدًا، يرجى كتابة سؤال أوضح.",
            "retrieved_titles": [],
            "retrieved_chunks": [],
            "hybrid_scores": []
        }

    try:
        # Attempt normal Hybrid Search retrieval
        search_results = hybrid_search(
            query=user_question,
            faiss_index=faiss_index,
            embedding_model=embedding_model,
            bm25=bm25,
            tokenized_documents=tokenized_documents,
            chunks_df=chunks_df,
            k=k,
            alpha=0.7
        )
    except Exception as error:
        print("⚠️ Retrieval failed. Reason:", str(error))
        return {
            "answer": "حدث خطأ أثناء البحث عن معلومات ذات صلة.",
            "retrieved_titles": [],
            "retrieved_chunks": [],
            "hybrid_scores": []
        }

    # Case 3: No retrieved chunks
    if search_results.empty:
        print("⚠️ No relevant chunks were retrieved.")
        return {
            "answer": "لم أجد معلومات كافية للإجابة اعتمادًا على القصص الموجودة.",
            "retrieved_titles": [],
            "retrieved_chunks": [],
            "hybrid_scores": []
        }

    # Case 4: Model generation failure
    try:
        result = generate_rag_answer(user_question, k=k, max_new_tokens=max_new_tokens)
        return result

    except Exception as error:
        print("⚠️ Model generation failed. Reason:", str(error))
        return {
            "answer": "حدث خطأ أثناء توليد الإجابة. يرجى المحاولة مرة أخرى.",
            "retrieved_titles": search_results["title"].tolist(),
            "retrieved_chunks": search_results["chunk_text"].tolist(),
            "hybrid_scores": search_results["hybrid_score"].tolist()
        }

## 15. Example Queries

Run the full RAG pipeline on a range of example questions — answerable, broad, and
out-of-scope — including edge cases like empty or very short queries, to demonstrate the
system end-to-end.

In [ ]:
# Prepare at least 15 test questions covering different scenarios

rag_test_questions = [
    # Answerable from one story
    "أين كانت الساعة معلقة؟",
    "ماذا حدث للساعة في يوم من الأيام؟",
    "من هي الفتاة التي أصلحت الساعة؟",
    "ماذا فعل أهل القرية عندما توقفت الساعة؟",
    "ما الدرس الذي تعلمه أهل القرية من توقف الساعة؟",

    # Requires broader retrieval across the dataset
    "ما هي القصص التي تتحدث عن الصداقة؟",
    "أعطني قصة عن الشجاعة",
    "هل هناك قصة عن الحيوانات في الغابة؟",
    "ما هي القصص التي تتحدث عن دروس وحكم؟",
    "أعطني قصة عن مغامرة",

    # No answer expected (out of scope / not in dataset)
    "من هو رئيس الجمهورية الحالي؟",
    "ما هي عاصمة اليابان؟",
    "كيف يمكنني طهي طبق الكشري؟",
    "ما هو سعر الذهب اليوم؟",
    "من فاز بكأس العالم لكرة القدم؟"
]

print("Prepared", len(rag_test_questions), "test questions for RAG evaluation.")

In [ ]:
# Run the full RAG pipeline on every test question and print results

for question in rag_test_questions:
    print("=" * 100)
    print("Question:", question)

    result = generate_rag_answer(question, k=3)

    print("\nRetrieved Story Title(s):", result["retrieved_titles"])
    print("Hybrid Score(s):", [round(s, 4) for s in result["hybrid_scores"]])
    print("\nGenerated Answer:", result["answer"])

print("=" * 100)
print("\nRAG system testing completed!")

In [ ]:
# Test all error handling cases

error_test_cases = [
    "",              # empty query
    "ا",              # very short query
    "أين كانت الساعة معلقة؟"  # normal, valid query (should work fine)
]

for test_case in error_test_cases:
    print("=" * 80)
    print("Testing input:", repr(test_case))

    result = safe_generate_rag_answer(test_case, k=3)

    print("Answer:", result["answer"])

print("=" * 80)
print("\nError handling testing completed!")

## 16. Save Project Artifacts

Persist the key outputs produced by this notebook (cleaned dataset, chunk embeddings,
chunk table, ground-truth set, and evaluation results) so they can be reused without
re-running the full pipeline.

In [ ]:
# Save the cleaned DataFrame to a CSV file
output_path = "stories_clean.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Cleaned dataset saved successfully!")
print("Saved to:", output_path)

In [ ]:
# Verify by reloading the saved CSV file and checking its shape and columns
verify_df = pd.read_csv(output_path)

print("Verification - Reloaded CSV shape:", verify_df.shape)
print("Verification - Reloaded CSV columns:", list(verify_df.columns))

print("\nSample reloaded row:")
print(verify_df.iloc[0])

In [ ]:
# Save embeddings to a .npy file for reuse later
embeddings_path = "embeddings.npy"

np.save(embeddings_path, embeddings)

print("Embeddings saved successfully!")
print("Saved to:", embeddings_path)

In [ ]:
# Save additional project artifacts for reuse and reproducibility
chunks_output_path = "chunks.csv"
chunks_df.to_csv(chunks_output_path, index=False, encoding="utf-8-sig")

ground_truth_output_path = "ground_truth.csv"
ground_truth_df.to_csv(ground_truth_output_path, index=False, encoding="utf-8-sig")

evaluation_output_path = "evaluation_results.csv"
evaluation_df.to_csv(evaluation_output_path, index=False, encoding="utf-8-sig")

print("Saved artifacts:")
print("-", chunks_output_path)
print("-", ground_truth_output_path)
print("-", evaluation_output_path)

## Conclusion

This notebook implemented a complete Arabic RAG system from scratch:

- Loaded and cleaned a raw JSON dataset of Arabic short stories.
- Built a `retrieval_text` field and split it into overlapping chunks.
- Indexed the chunks with both **BM25** (keyword) and **FAISS** (semantic embeddings)
  retrieval, and combined them into a single **Hybrid Search** ranking.
- Evaluated retrieval quality against a hand-built ground-truth query set using
  Precision@k, Recall@k, Hit Rate@k, and Mean Reciprocal Rank.
- Loaded an open-source LLM and built a full **RAG pipeline** that answers user questions
  strictly from retrieved story context, with graceful error handling for invalid inputs.
- Saved all key intermediate and final artifacts (cleaned dataset, embeddings, chunks,
  ground truth, and evaluation results) for reuse.

The result is an end-to-end, reproducible pipeline that can be extended with a larger
dataset, additional evaluation queries, or a more capable LLM.